# S5–S7 流程

本 Notebook 演示推理、分类导出以及 QC 报告生成。

In [ ]:
import sys
sys.path.append(r"d:\Desktop\Spike sort")


import numpy as np
import torch

from Main.S1 import load_input
from Main.S4 import build_simple_mlp
from Main.S5 import predict
from Main.S6 import split_by_class, export_to_excel
from Main.S7 import generate_qc_report



In [ ]:
# 读取数据
path = "./Data/Process data/c57_v_c57_test1_wave_events.npz"
data = np.load(path, allow_pickle=True)
X = data["X"]
T0 = data["T0"] 



In [ ]:
# 读取已训练模型参数
model_path = "./Model/simple_mlp.pt"
num_classes = 2  
model = build_simple_mlp(input_dim=X.shape[2], num_classes=num_classes)
model.load_state_dict(torch.load(model_path, map_location="cpu"))
model.eval()


In [ ]:
# 将 X 转为 (N, t) 做推理
X_flat = X.reshape(-1, X.shape[2])
pred_flat = predict(model, X_flat)
pred = pred_flat.reshape(X.shape[0], X.shape[1])

bundle = split_by_class(X, T0, pred)
export_to_excel(bundle, "./Result/Export")

qc = generate_qc_report(X, pred)
qc